In [1]:
# !pip uninstall -y bitsandbytes
# !pip install -U bitsandbytes>=0.46.1
# !pip install -U accelerate transformers

In [1]:
import torch

print(torch.cuda.get_device_name())
print(torch.cuda.get_device_capability())

NVIDIA GeForce RTX 5060 Ti
(12, 0)


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5060 Ti with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5060 Ti GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [2]:
import torch
import transformers

print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Transformers: 5.3.0
Torch: 2.5.1+cu121
CUDA: True
GPU: NVIDIA GeForce RTX 5060 Ti


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = r"D:\khmer_spell_check\spellcheck_textsummarize\Qwen3-4B-Khmer-text-summarizer-20260310T014415Z-1-002"

device = "cpu"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.float16, device_map=device, trust_remote_code=True
)

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_PATH, dtype=torch.float16, device_map="auto", trust_remote_code=True
# )

model.eval()

print("Model loaded successfully!")

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
def generate_summary(text, max_length_ratio=0.3, max_new_tokens=1024):
    """
    Generate Khmer summary with dynamic output length.
    """

    # Estimate desired output size
    input_ids = tokenizer(text)["input_ids"]
    input_token_count = len(input_ids)

    target_max_tokens = int(input_token_count * max_length_ratio)
    target_max_tokens = min(target_max_tokens, max_new_tokens)

    prompt = f"""<|im_start|>user
    សូមសង្ខេបអត្ថបទខាងក្រោមជាអត្ថបទខ្លី និងច្បាស់លាស់។
    {text}
    <|im_end|>
    <|im_start|>assistant
    """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=target_max_tokens,
            temperature=0.3,
            top_p=0.9,
            top_k=40,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract assistant part
    if "<|im_start|>assistant" in generated_text:
        assistant_response = (
            generated_text.split("<|im_start|>assistant")[1]
            .split("<|im_end|>")[0]
            .strip()
        )
    else:
        assistant_response = generated_text.strip()

    return assistant_response

In [11]:
long_text = """ប្រធានាធិបតីអាមេរិក លោក ដូណាល់ ត្រាំ បានគិតគូរជាយូរណាស់មកហើយចង់ឱ្យសហរដ្ឋអាមេរិកគ្រប់គ្រងដែនដី Greenland ជាគំនិតមួយដែលធ្វើឱ្យបណ្ដាមេដឹកនាំអឺរ៉ុប និងប្រជាជន Greenland ខឹងសម្បារផង និងសើចចំអកផង។ ប៉ុន្តែនៅបន្ទាប់ពីសហរដ្ឋអាមេរិកវាយប្រហារលើវ៉េណេស៊ុយអេឡា និងចាប់ខ្លួនលោក នីកូឡាស់ ម៉ាឌូរ៉ូ កាលពីថ្ងៃទី៣ ខែមករា វាបានបង្ហាញថាមហិច្ឆតារបស់លោក ត្រាំ ចង់គ្រប់គ្រងដែនដីក្នុងតំបន់អាក់ទិកមួយនេះហាក់មានភាពប្រាកដប្រជាជាងពេលណាៗទាំងអស់។
Greenland ដែលជាតំបន់ស្វយ័តមួយរបស់ដាណឺម៉ាក និងមានប្រជាជនរស់នៅ ៥៧០០០នាក់ បានទទួលឱ្យសហរដ្ឋអាមេរិកដាក់មូលដ្ឋានទ័ពមួយរួចទៅហើយ។ តែរដ្ឋបាលលោក ត្រាំចង់បាន សិទិ្ធអំណាចគ្រប់គ្រងបន្ថែមលើដែនដីមួយនេះ ដោយសារតែ Greenland មានទីភូមិសាស្ត្រយុទ្ធសាស្ត្រយ៉ាងសំខាន់នៅក្នុងតំបន់អាក់ទិកដើម្បីការពារផលប្រយោជន៍សន្តិសុខសហរដ្ឋអាមេរិក និង NATO។ «យើងត្រូវការ Greenland សម្រាប់ការការពារ»។ នេះជាអ្វីដែលលោក ត្រាំ បានប្រាប់កាសែតអាមេរិក The Atlantic កាលពីថ្ងៃទី៤ ខែមករា ជាមួយការអះអាងថា Greenland កំពុងត្រូវបានហ៊ុមព័ទ្ធដោយកប៉ាល់ចិន និងរុស្ស៉ី។ Greenland ក៏សម្បូរផងដែរធនធានធម្មជាតិ ក្នុងនោះរួមមានឧស្ម័នធម្មជាតិ ប្រេង និងរ៉ែដែលត្រូវបានប្រើក្នុងបច្ចេកវិទ្យា និងយោធាដែលកំពុង ក្លាយជាចំណុចក្ដៅមួយក្នុងសង្រ្គាមពាណិជ្ជកម្មរវាងសហរដ្ឋអាមេរិក និងចិន។
តែទោះជាបែបនេះក្ដី រដ្ឋបាលលោក ត្រាំ កំពុងប្រឈមនឹងការប្រឆាំងជំទាស់យ៉ាងខ្លាំងពីបណ្ដាមេដឹកនាំ NATO ក្នុងនោះមានទាំងដាណឺម៉ាដែលនៅតែមានសិទ្ធិអំណាចចាត់ចែងគោលនយោបាយ ការពារជាតិ និងការបរទេសរបស់ Greenland។ នៅក្នុងសេចក្ដីថ្លែងការណ៍មួយកាលពីថ្ងៃទី៦ ខែមករា មេដឹកនាំដាណឺម៉ាក បារាំង អាល្លឺម៉ង់ អ៉ីតាលី ប៉ូឡូញ និងចក្រភពអង់គ្លេសបាននិយាយថា មានតែប្រជាជនដាណឺម៉ាក និងប្រជាជន Greenland តែប៉ុណ្ណោះដែលអាចសម្រេចចិត្តលើបញ្ហាពាក់ព័ន្ធនឹងដាណឺម៉ាក និង Greenland។ ដូច្នេះតើលោក ត្រាំ អាចមានវិធីសាស្ត្រអ្វីខ្លះដើម្បី គ្រប់គ្រង Greenland?
"""

summary = generate_summary(long_text, max_length_ratio=0.3, max_new_tokens=1024)
print("Summary:\n", summary)

Both `max_new_tokens` (=551) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summary:
 user
    សូមសង្ខេបអត្ថបទខាងក្រោមជាអត្ថបទខ្លី និងច្បាស់លាស់។
    ប្រធានាធិបតីអាមេរិក លោក ដូណាល់ ត្រាំ បានគិតគូរជាយូរណាស់មកហើយចង់ឱ្យសហរដ្ឋអាមេរិកគ្រប់គ្រងដែនដី Greenland ជាគំនិតមួយដែលធ្វើឱ្យបណ្ដាមេដឹកនាំអឺរ៉ុប និងប្រជាជន Greenland ខឹងសម្បារផង និងសើចចំអកផង។ ប៉ុន្តែនៅបន្ទាប់ពីសហរដ្ឋអាមេរិកវាយប្រហារលើវ៉េណេស៊ុយអេឡា និងចាប់ខ្លួនលោក នីកូឡាស់ ម៉ាឌូរ៉ូ កាលពីថ្ងៃទី៣ ខែមករា វាបានបង្ហាញថាមហិច្ឆតារបស់លោក ត្រាំ ចង់គ្រប់គ្រងដែនដីក្នុងតំបន់អាក់ទិកមួយនេះហាក់មានភាពប្រាកដប្រជាជាងពេលណាៗទាំងអស់។
Greenland ដែលជាតំបន់ស្វយ័តមួយរបស់ដាណឺម៉ាក និងមានប្រជាជនរស់នៅ ៥៧០០០នាក់ បានទទួលឱ្យសហរដ្ឋអាមេរិកដាក់មូលដ្ឋានទ័ពមួយរួចទៅហើយ។ តែរដ្ឋបាលលោក ត្រាំចង់បាន សិទិ្ធអំណាចគ្រប់គ្រងបន្ថែមលើដែនដីមួយនេះ ដោយសារតែ Greenland មានទីភូមិសាស្ត្រយុទ្ធសាស្ត្រយ៉ាងសំខាន់នៅក្នុងតំបន់អាក់ទិកដើម្បីការពារផលប្រយោជន៍សន្តិសុខសហរដ្ឋអាមេរិក និង NATO។ «យើងត្រូវការ Greenland សម្រាប់ការការពារ»។ នេះជាអ្វីដែលលោក ត្រាំ បានប្រាប់កាសែតអាមេរិក The Atlantic កាលពីថ្ងៃទី៤ ខែមករា ជាមួយការអះអាងថា Greenland កំពុងត្រូវបានហ៊ុមព័ទ្ធដោយកប៉ាល់ចិន និងរុស្ស៉ី។ Gr